# Heart Disease Risk Prediction - Placement Ready ML Notebook

This notebook contains an upgraded ML workflow with EDA, model comparison, evaluation, and model export.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import joblib

In [ ]:
data = pd.read_csv('../data/heart_data.csv')
data.head()

In [ ]:
print('Shape:', data.shape)
print('\nMissing values:')
print(data.isnull().sum())
print('\nTarget distribution:')
print(data['target'].value_counts())

In [ ]:
data.describe()

In [ ]:
X = data.drop('target', axis=1)
y = data['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train.shape, X_test.shape

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
    ]),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced"),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(probability=True, class_weight="balanced", random_state=42))
    ])
}

results = []

for model_name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X, y, cv=cv, scoring="f1")

    results.append({
        "model": model_name,
        "accuracy": accuracy_score(y_test, predictions),
        "precision": precision_score(y_test, predictions),
        "recall": recall_score(y_test, predictions),
        "f1_score": f1_score(y_test, predictions),
        "roc_auc": roc_auc_score(y_test, probabilities),
        "cv_f1_mean": cv_scores.mean(),
        "cv_f1_std": cv_scores.std()
    })

results_df = pd.DataFrame(results).sort_values("f1_score", ascending=False)
results_df

In [ ]:
best_model_name = results_df.iloc[0]["model"]
best_model = models[best_model_name]
best_model.fit(X_train, y_train)

predictions = best_model.predict(X_test)

print("Best Model:", best_model_name)
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, predictions))
print("\nClassification Report:")
print(classification_report(y_test, predictions))

In [ ]:
joblib.dump(best_model, "../models/heart_disease_model.pkl")
results_df.to_csv("../reports/model_comparison.csv", index=False)

print("Model and report saved successfully.")

## Unique Features Added

- Multiple model comparison
- Cross-validation
- Risk probability
- Low / Moderate / High risk category in Streamlit app
- Personalized health suggestions
- Feature guide
- GitHub-ready structure
- Resume-ready documentation